#Init step

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType

#Read from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data transformations

In [0]:
df.display()

#Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name, trim(col(field.name)))
df.display()

#Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.display()

#Remove records where customer id is null

In [0]:
df = df.filter(col("cst_id").isNotNull())

#Rename columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()

#Write into Silver table

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")


In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10